# 03 · Chạy Qwen 7B (C0 và RAG) trên bộ pilot (Tuần 3 · TV2)

**Chạy ở đâu:** Google Colab, **Runtime → Change runtime type → T4 GPU**, rồi **Runtime → Run all**.

**Làm gì:** phục vụ `Qwen/Qwen2.5-7B-Instruct-AWQ` bằng vLLM, chạy `rag.run_inference` cho C0 (không truy xuất) và C2 (RAG: BGE-M3 + chia Điều–Khoản, top-5) trên `data/benchmark/pilot_80_v0.3.jsonl`. Ngữ cảnh RAG đã được truy xuất sẵn trên máy TV2 (`outputs/contexts/`), nên Colab chỉ cần sinh câu trả lời.

**Kết quả:** ô cuối tải về `qwen_pilot_outputs.zip`. Gửi file này cho TV2 (hoặc giải nén vào thư mục dự án).

Bị ngắt giữa chừng: chạy lại từ ô 2. `run_inference` tự chạy tiếp các câu còn thiếu nếu thư mục `repo/outputs` còn.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
%%capture
!pip install vllm
!git clone -b an-huy https://github.com/yuhna111/NEXUS2026-FeelTheBeat repo
!pip install -e repo

In [ ]:
import subprocess, time, requests

MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # trùng snapshot của qwen-7b trong configs/models.yaml
log = open("vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--port", "8000",
     "--max-model-len", "4096", "--gpu-memory-utilization", "0.90"],
    stdout=log, stderr=subprocess.STDOUT,
)

t0 = time.time()
while True:
    if server.poll() is not None:
        raise RuntimeError("vLLM dừng sớm — xem vllm.log:\n" + open("vllm.log").read()[-3000:])
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
print(f"Máy chủ sẵn sàng sau {time.time() - t0:.0f} giây")

In [ ]:
import os
os.chdir("repo")
os.environ["VLLM_BASE_URL"] = "http://localhost:8000/v1"
os.environ["LLM_CACHE_DIR"] = ".llm_cache"
BENCH = "data/benchmark/pilot_80_v0.3.jsonl"
!ls outputs/contexts/pilot_80_v0.3/

In [ ]:
# C0: không truy xuất, temperature 0, 1 lần (~5–10 phút)
!python -m rag.run_inference --config C0 --bench $BENCH --models qwen-7b

In [ ]:
# C2: RAG (ngữ cảnh truy xuất sẵn), temperature 0, 1 lần (~5–10 phút)
!python -m rag.run_inference --config C2 --bench $BENCH --models qwen-7b

In [ ]:
# Đo tốc độ sinh trên các lượt vừa chạy
import json, glob
for f in sorted(glob.glob("outputs/pilot_80_v0.3/*/qwen-7b*.jsonl")):
    rows = [json.loads(l) for l in open(f, encoding="utf-8")]
    toks = [r["usage"].get("completion_tokens", 0) for r in rows]
    print(f, len(rows), "câu ·", sum(toks), "token ra")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
server.terminate()
!zip -qr /content/qwen_pilot_outputs.zip outputs/pilot_80_v0.3 .llm_cache
from google.colab import files
files.download("/content/qwen_pilot_outputs.zip")